<a href="https://colab.research.google.com/github/Adhishiyaa/git_notes/blob/main/ATM_PIN_setup_interface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tkinter as tk
from tkinter import messagebox


class ATMPinSetup:
    def __init__(self, root):
        self.root = root
        root.title("ATM PIN Setup")
        root.geometry("460x640")
        root.resizable(False, False)
        root.configure(bg="#dfe6ee")

        tk.Label(
            root, text="ATM PIN Setup", font=("Arial", 18, "bold"),
            bg="#dfe6ee", fg="#12355b"
        ).pack(pady=(12, 6))

        self.build_account_frame()
        self.build_keypad_frame()
        self.build_output_frame()

        # The keypad types into whichever PIN box was clicked last.
        self.active_entry = self.pin_entry

    # ------------------------------------------------------------------
    # Account details: RAISED frame, widgets positioned with place()
    # ------------------------------------------------------------------
    def build_account_frame(self):
        frame = tk.Frame(self.root, width=420, height=190,
                         relief=tk.RAISED, bd=5, bg="#f4f7fb")
        frame.pack(pady=6)
        frame.pack_propagate(False)   # keep the fixed size for place()

        tk.Label(frame, text="Account Details", font=("Arial", 11, "bold"),
                 bg="#f4f7fb").place(x=10, y=6)

        tk.Label(frame, text="Account Holder:", bg="#f4f7fb").place(x=15, y=40)
        tk.Label(frame, text="Account Number:", bg="#f4f7fb").place(x=15, y=75)
        tk.Label(frame, text="Enter PIN:", bg="#f4f7fb").place(x=15, y=110)
        tk.Label(frame, text="Confirm PIN:", bg="#f4f7fb").place(x=15, y=145)

        self.name_entry = tk.Entry(frame, width=26)
        self.name_entry.place(x=150, y=40)

        self.acc_entry = tk.Entry(frame, width=26)
        self.acc_entry.place(x=150, y=75)

        # show="*" masks the typed digits
        self.pin_entry = tk.Entry(frame, width=26, show="*")
        self.pin_entry.place(x=150, y=110)

        self.confirm_entry = tk.Entry(frame, width=26, show="*")
        self.confirm_entry.place(x=150, y=145)

        self.pin_entry.bind("<FocusIn>", lambda e: self.set_active(self.pin_entry))
        self.confirm_entry.bind("<FocusIn>", lambda e: self.set_active(self.confirm_entry))

    def set_active(self, entry):
        self.active_entry = entry

    # ------------------------------------------------------------------
    # Keypad: SUNKEN frame, each key in its own bordered cell, grid()
    # ------------------------------------------------------------------
    def build_keypad_frame(self):
        pad = tk.Frame(self.root, relief=tk.SUNKEN, bd=5, bg="#c9d3df")
        pad.pack(pady=6)

        keys = [
            "1", "2", "3",
            "4", "5", "6",
            "7", "8", "9",
            "Clear", "0", "Del",
        ]

        for i, key in enumerate(keys):
            row, col = divmod(i, 3)
            cell = tk.Frame(pad, relief=tk.RAISED, bd=3, bg="#e8edf3")
            cell.grid(row=row, column=col, padx=4, pady=4)

            tk.Button(
                cell, text=key, width=7, height=2, font=("Arial", 11, "bold"),
                command=lambda k=key: self.key_press(k)
            ).pack()

    def key_press(self, key):
        entry = self.active_entry
        if key == "Clear":
            entry.delete(0, tk.END)
        elif key == "Del":
            entry.delete(len(entry.get()) - 1, tk.END)
        else:
            if len(entry.get()) < 4:      # PIN is 4 digits
                entry.insert(tk.END, key)

    # ------------------------------------------------------------------
    # Output area + button that reads the form and updates the Text widget
    # ------------------------------------------------------------------
    def build_output_frame(self):
        tk.Button(
            self.root, text="Set PIN", font=("Arial", 11, "bold"),
            bg="#1f6feb", fg="white", width=18, command=self.set_pin
        ).pack(pady=6)

        out = tk.Frame(self.root, relief=tk.SUNKEN, bd=4)
        out.pack(pady=4)

        self.output = tk.Text(out, width=48, height=6, state=tk.DISABLED,
                              bg="#fffef5")
        self.output.pack()

    def write_output(self, message):
        self.output.config(state=tk.NORMAL)
        self.output.delete("1.0", tk.END)
        self.output.insert(tk.END, message)
        self.output.config(state=tk.DISABLED)

    def set_pin(self):
        name = self.name_entry.get().strip()
        acc = self.acc_entry.get().strip()
        pin = self.pin_entry.get()
        confirm = self.confirm_entry.get()

        if not name or not acc:
            messagebox.showwarning("Missing details",
                                   "Enter the account holder and account number.")
            return
        if not acc.isdigit():
            messagebox.showwarning("Invalid account", "Account number must be digits only.")
            return
        if len(pin) != 4 or not pin.isdigit():
            messagebox.showwarning("Invalid PIN", "PIN must be exactly 4 digits.")
            return
        if pin != confirm:
            messagebox.showerror("Mismatch", "PIN and Confirm PIN do not match.")
            return

        masked_acc = "*" * (len(acc) - 4) + acc[-4:] if len(acc) > 4 else acc
        self.write_output(
            "PIN set successfully!\n"
            f"Account Holder : {name}\n"
            f"Account Number : {masked_acc}\n"
            f"PIN            : {'*' * len(pin)}"
        )

        for e in (self.pin_entry, self.confirm_entry):
            e.delete(0, tk.END)


if__name__=="__main__":
   root = tk.Tk()
    ATMPinSetup(root)
    root.mainloop()